In [ ]:
import numpy as np
import matplotlib.pyplot as plt

variant = 1
lam = 0.20                # λ варіанта 1 (Київ)
true_mean = 1 / lam       # 5.0 хв — відоме наперед теоретичне E[X]

rng = np.random.default_rng(variant)   # фіксація для відтворюваності на всі завдання

In [ ]:
print("Теоретичне E[X] = 1/λ = {:.2f} хв\n".format(true_mean))
print("{:>8} | {:>10} | {:>14}".format("n", "x̄ (mean)", "|x̄ − E[X]|"))
print("-" * 38)

for n in [5, 50, 500, 20_000]:
    sample = rng.exponential(scale=1 / lam, size=n)
    m = sample.mean()
    print("{:>8} | {:>10.4f} | {:>14.4f}".format(n, m, abs(m - true_mean)))

In [ ]:
n_repeats = 10_000

# кожен рядок матриці — окрема вибірка; .mean(axis=1) рахує середнє рядка без циклу for
means_n10  = rng.exponential(scale=1 / lam, size=(n_repeats, 10)).mean(axis=1)
means_n300 = rng.exponential(scale=1 / lam, size=(n_repeats, 300)).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(means_n10, bins=40, density=True, color="steelblue", edgecolor="white")
axes[0].axvline(true_mean, color="crimson", linestyle="--", label="E[X]=5.0")
axes[0].set_title("n=10, std(x̄)={:.3f}".format(means_n10.std()))
axes[0].set_xlabel("x̄ (хв)")
axes[0].legend()

axes[1].hist(means_n300, bins=40, density=True, color="seagreen", edgecolor="white")
axes[1].axvline(true_mean, color="crimson", linestyle="--", label="E[X]=5.0")
axes[1].set_title("n=300, std(x̄)={:.3f}".format(means_n300.std()))
axes[1].set_xlabel("x̄ (хв)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
n_repeats = 10_000
rows = []

for n in [10, 30, 100, 300]:
    se_theory = (1 / lam) / np.sqrt(n)          # теоретична SE = (1/λ)/√n
    sample_means = rng.exponential(scale=1 / lam, size=(n_repeats, n)).mean(axis=1)
    se_measured = sample_means.std()            # виміряний розкид 10 000 реалізацій x̄
    rows.append((n, se_theory, se_measured))

print("{:>6} | {:>16} | {:>16}".format("n", "теоретична SE", "виміряна std"))
print("-" * 44)
for n, st, sm in rows:
    print("{:>6} | {:>16.4f} | {:>16.4f}".format(n, st, sm))

In [ ]:
se_25  = (1 / lam) / np.sqrt(25)
se_100 = (1 / lam) / np.sqrt(100)

print("SE(n=25)  = {:.4f}".format(se_25))
print("SE(n=100) = {:.4f}".format(se_100))
print("Співвідношення SE(25)/SE(100) = {:.2f}".format(se_25 / se_100))

In [ ]:
means_few  = rng.exponential(scale=1 / lam, size=(50, 10)).mean(axis=1)
means_many = rng.exponential(scale=1 / lam, size=(10_000, 10)).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(means_few, bins=15, density=True, color="orange", edgecolor="white")
axes[0].set_title("n=10, n_repeats=50 (замало)")
axes[0].set_xlabel("x̄ (хв)")

axes[1].hist(means_many, bins=40, density=True, color="steelblue", edgecolor="white")
axes[1].set_title("n=10, n_repeats=10000 (багато)")
axes[1].set_xlabel("x̄ (хв)")

plt.tight_layout()
plt.show()

## Контрольні питання

### 1. Закон великих чисел vs центральна гранична теорема

- **Закон великих чисел** відповідає на питання **«куди прямує x̄»**: він гарантує, що вибіркове середнє x̄ збігається до теоретичного E[X] зі зростанням n (про наближення числа до числа).
- **Центральна гранична теорема** відповідає на питання **«яка форма й розкид у x̄»**: вона описує розподіл самого x̄ — стверджує, що він набуває нормальної форми з середнім E[X] і стандартним відхиленням σ/√n, незалежно від форми вихідної сукупності.

Тобто ЗВЧ — про границю (куди прямує), ЦГТ — про розподіл навколо цієї границі (як розкидані значення x̄).

### 2. Навіщо вихідний розподіл сильно скошений, а не нормальний

Якби вихідна сукупність одразу була нормальною, то x̄ був би нормальним при будь-якому n (навіть n = 2) — і ЦГТ нічого нового не показала б. Скошений (експоненційний) розподіл дозволяє побачити **перехід**: при малому n розподіл x̄ ще успадковує скошеність, а при великому n — стає дзвоноподібним. Це наочно демонструє силу теореми: нормальність x̄ виникає завдяки засередненню, а не завдяки формі вихідних даних.

### 3. Чому SE ∝ √n, а не n

Бо SE = σ/√n — n стоїть під коренем (це наслідок того, що дисперсії незалежних доданків додаються, а σ — квадратний корінь із дисперсії). Практично: щоб зменшити похибку **вдвічі**, треба збільшити n **вчетверо** (бо √4 = 2). Тобто кожен наступний крок уточнення коштує дедалі дорожче — закон убуваючої віддачі.